## Read Bronze


In [0]:
orders_df = spark.table("de_workspace26.bronze_aditya_sah_assessment.orders")
customers_df = spark.table("de_workspace26.bronze_aditya_sah_assessment.customers")
products_df = spark.table("de_workspace26.bronze_aditya_sah_assessment.products")

## Deduplicate

In [0]:
orders_df = orders_df.dropDuplicates(["order_id"])

## Cast Date + Add Revenue

In [0]:
from pyspark.sql.functions import col

orders_df = orders_df.withColumn(
    "order_date",
    col("order_date").cast("date")
).withColumn(
    "revenue",
    col("quantity") * col("unit_price")
)

### Window Function (Cumulative Revenue)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum as _sum

window_spec = Window.partitionBy("customer_id").orderBy("order_date")

orders_df = orders_df.withColumn(
    "cumulative_revenue",
    _sum("revenue").over(window_spec)
)

In [0]:
orders_df = orders_df.alias("o")
customers_df = customers_df.alias("c")
products_df = products_df.alias("p")

### Join with Customers + Products

In [0]:
orders_enriched_df = orders_df \
    .join(customers_df, col("o.customer_id") == col("c.customer_id"), "left") \
    .join(products_df, col("o.product_id") == col("p.product_id"), "left")

### Select Required Columns (Clean Output)

In [0]:
from pyspark.sql.functions import col

orders_enriched_df = orders_enriched_df.select(
    col("o.order_id"),
    col("o.customer_id"),
    col("o.product_id"),
    col("o.order_date"),
    col("o.quantity"),
    col("o.unit_price"),
    col("o.revenue"),
    col("o.cumulative_revenue"),
    col("o.status"),
    col("o.region"),

    col("c.city"),
    col("c.loyalty_tier"),

    col("p.product_name"),
    col("p.category")
)

### Write Silver Orders Table (Partitioned)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS de_workspace26.silver_aditya_sah_assessment;

In [0]:
orders_enriched_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("region") \
    .saveAsTable("de_workspace26.silver_aditya_sah_assessment.orders")

## SCD TYPE 2 — Customers

In [0]:
from pyspark.sql.functions import current_date, lit

customers_df = spark.table("de_workspace26.bronze_aditya_sah_assessment.customers")

customers_scd_df = customers_df \
    .withColumn("is_current", lit(True)) \
    .withColumn("effective_start_date", current_date()) \
    .withColumn("effective_end_date", lit(None).cast("date"))

customers_scd_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("de_workspace26.silver_aditya_sah_assessment.customers")

In [0]:
%sql
MERGE INTO de_workspace26.silver_aditya_sah_assessment.customers tgt
USING de_workspace26.bronze_aditya_sah_assessment.customers src
ON tgt.customer_id = src.customer_id AND tgt.is_current = true

WHEN MATCHED AND tgt.loyalty_tier <> src.loyalty_tier THEN
  UPDATE SET 
    tgt.is_current = false,
    tgt.effective_end_date = current_date()

WHEN NOT MATCHED THEN
  INSERT (
    customer_id,
    name,
    email,
    city,
    loyalty_tier,
    signup_date,
    _ingested_at,
    _source_file,
    is_current,
    effective_start_date,
    effective_end_date
  )
  VALUES (
    src.customer_id,
    src.name,
    src.email,
    src.city,
    src.loyalty_tier,
    src.signup_date,
    src._ingested_at,
    src._source_file,
    true,
    current_date(),
    NULL
  )

In [0]:
%sql
ALTER TABLE de_workspace26.silver_aditya_sah_assessment.orders
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
DESCRIBE HISTORY de_workspace26.silver_aditya_sah_assessment.orders;

In [0]:
%sql
UPDATE de_workspace26.silver_aditya_sah_assessment.orders
SET quantity = quantity + 1
WHERE order_id IS NOT NULL;

In [0]:
%sql
SELECT *
FROM table_changes(
  'de_workspace26.silver_aditya_sah_assessment.orders',
  3   
);

In [0]:
%sql
UPDATE de_workspace26.silver_aditya_sah_assessment.orders
SET quantity = quantity + 1
WHERE order_id IS NOT NULL;